In [7]:
import numpy as np
from pathlib import Path
import prism

from imagematerials.factory import ModelFactory
from imagematerials.model import RestOf
from imagematerials.preprocessing import get_preprocessing_data
from imagematerials.rest_of.preprocessing.main import fit_all_materials_save_corrseponding_input_data

from imagematerials.rest_of.preprocessing.resource_efficiency_measures import calcualte_resource_efficiency_measures

In [ ]:
# REFIT DATA
refit = False

if refit == True:
    fit_all_materials_save_corrseponding_input_data(path_input_data=Path("../data/raw/rest-of"),
                                                    path_input_data_image=Path("../data/raw/image"))
    
# REFIT EFFICIENCY MEASURES
refit_efficiency_measures = False
if refit_efficiency_measures == True:
    calcualte_resource_efficiency_measures()

SSP2_baseline
copper group_3 assigned to class_ 18
copper group_3 assigned to class_ 21
copper group_3 assigned to class_ 22
copper all_regions assigned to class_ 10
copper all_regions assigned to class_ 12
steel all_regions assigned to class_ 14
steel all_regions assigned to class_ 15
steel all_regions assigned to class_ 16
steel group_8 assigned to class_ 4
steel group_8 assigned to class_ 25
steel group_8 assigned to class_ 26
steel group_8 assigned to class_ 18
steel group_8 assigned to class_ 22
cement all assigned to class_ 15
cement all assigned to class_ 16
cement all assigned to class_ 17
cement all assigned to class_ 22
cement group_2 assigned to class_ 18
cement group_2 assigned to class_ 25
sand_gravel_crushed_rock all_regions assigned to class_ 5
sand_gravel_crushed_rock group_10 assigned to class_ 8
sand_gravel_crushed_rock group_10 assigned to class_ 9
sand_gravel_crushed_rock group_10 assigned to class_ 25
sand_gravel_crushed_rock group_10 assigned to class_ 26
limeston

In [9]:
# Define the complete timeline, including historic tail

YEAR_START  = 1971    # start year of the simulation period
YEAR_END    = 2100    # end year of the calculations
YEAR_OUT    = 2100    # year of output generation = last year of reporting

time_start = 1971
complete_timeline = prism.Timeline(time_start, YEAR_END, 1)
simulation_timeline = prism.Timeline(YEAR_START, YEAR_OUT, 1)

In [10]:
scenario_list = {"SSP2_baseline":("SSP2_baseline", None)}

In [11]:
# Define paths
scenario_base_path = Path("..", "data", "raw")

In [12]:
# Run simulations for all scenarios

from mypyc.__main__ import base_path


model_runs = {} # to store outputs for all scenarios

for scen_id, (climate_scen, circular_scen) in scenario_list.items():
    print(f"Running scenario: {scen_id}")
    print(f"Circular economy scenario: {circular_scen}")
    
    # needed for electricity
    scen = climate_scen.split("_")[0]
    variant = "_".join(climate_scen.split("_")[1:])
    
    climate_policy_scenario_dir = scenario_base_path / Path("image", climate_scen)
    if circular_scen is not None:
        circular_economy_scenario_dirs ={
                scenario: scenario_base_path / Path("circular_economy_scenarios", scenario) for scenario in circular_scen
            }
    else:
        circular_economy_scenario_dirs = None

# Get preprocessing data for all sectors and adjust material indices
# #REST OF
    rest_sector = get_preprocessing_data(sector="rest_of",base_dir=scenario_base_path,
                                         climate_policy_scenario_dir=climate_policy_scenario_dir,
                                         circular_economy_scenario_dirs=circular_economy_scenario_dirs,
                                         scenario_name=climate_scen)
    

# Build and run the model
    factory = ModelFactory(rest_sector,complete_timeline
                           ).add(RestOf)                

    model = factory.finish() # create the model
    model_runs[climate_scen] = model
    
    # Run the simulation (suppressing warnings)
    import warnings
    with warnings.catch_warnings():
        warnings.filterwarnings("ignore")
        model.simulate(simulation_timeline) 
    
    print(f"Finished {scen_id}")

Running scenario: SSP2_baseline
Circular economy scenario: None
Finished SSP2_baseline
